# OpenAI ToolSearch 구현 — tools 배열을 고정해서 캐시 미스 0 만들기

도구가 수십 개 이상이면 전부 `tools`에 넣기엔 토큰이 아깝습니다.
그래서 "필요할 때 검색해서 쓰는" 방식(ToolSearch)을 만드는데, 문제가 하나 있습니다.
**찾은 도구를 `tools` 배열에 추가하는 순간 프롬프트 캐시가 깨집니다.**

이 노트북은 그 문제를 피하는 설계를 구현합니다.
`tools`는 `tool_search`(찾기)와 `tool_invoke`(실행) 딱 2개로 고정하고,
검색 결과(도구 스키마)는 대화 내용 쪽으로 돌려줍니다. 대화는 뒤에 쌓이기만 하므로 캐시가 안 깨집니다.

이 노트북에서 하는 것:
1. ToolSearch 구현 (2단계 검색 + 실행 게이트웨이)
2. 실제 API 응답의 `cached_tokens`로 캐시가 유지되는지 확인 (실험 A)
3. 비교: 찾은 도구를 `tools`에 추가하는 구방식에서 캐시가 깨지는 것 확인 (실험 B)

설계 문서: `cc_agent_bible/md_group/openai-toolsearch-kv-cache.md`

## 배경 — 왜 tools를 안 바꾸면 캐시가 안 깨지나

OpenAI 프롬프트 캐싱의 규칙:
- 요청 앞부분이 이전 요청과 **완전히 같아야** 그 부분을 캐시에서 재사용합니다.
- 1024 토큰 이상부터 자동 적용되고, 128 토큰 단위로 캐시됩니다.
- 얼마나 재사용했는지는 응답의 `usage.input_tokens_details.cached_tokens`에 나옵니다.

요청은 이 순서로 만들어집니다:

```
[tools 배열] [시스템 지시문] [대화 내용 (계속 뒤에 쌓임)]
 ↑ 맨 앞. 여기가 바뀌면 뒤 전체가 캐시 미스
```

정리하면:

```
캐시를 깨는 것:     tools 배열 변경 (맨 앞이 바뀌므로)
캐시를 안 깨는 것:  대화 뒤에 붙는 모든 것 — 검색 결과, 스키마 텍스트, 실행 결과
```

그래서 전략은 하나입니다. **tools를 한 번 정하면 절대 바꾸지 않는다.**
도구 스키마는 `tool_search`의 결과(대화 내용)로 전달하고,
실행은 고정된 `tool_invoke`가 대신 해줍니다.

In [1]:
import json
import re
import time

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
MODEL = "gpt-5-nano"


## 1. 도구 레지스트리 — 클라이언트에만 있는 전체 도구 목록

도구 24개를 준비합니다. 이 목록은 **API에 보내지 않습니다.** (토큰 0)
모델은 이 도구들의 존재를 모르는 상태에서 시작하고, `tool_search`로만 찾을 수 있습니다.

- `description`은 "무엇을 하는 도구"보다 **"언제 쓰는 도구"** 중심으로 씁니다. 검색 품질이 여기서 결정됩니다.
- 실행 함수(handler)는 전부 가짜입니다. 실제 슬랙이나 캘린더에 연결하지 않습니다.
  이 노트북의 목적은 캐시 동작 확인이라서 실행 결과는 문자열이면 충분합니다.

In [2]:
def make_tool(name, description, params, handler=None, hint=""):
    """레지스트리 항목 생성. params = {인자이름: (타입, 설명)}, 전부 필수 인자로 취급.
    hint는 클로드코드의 searchHint에 해당하는 큐레이션 검색 힌트 (선택)."""
    properties = {k: {"type": t, "description": d} for k, (t, d) in params.items()}

    def default_handler(args, _name=name):
        return f"[가짜 실행 결과] {_name} 실행 완료 — 입력: {json.dumps(args, ensure_ascii=False)}"

    return {
        "name": name,
        "description": description,
        "search_hint": hint,
        "parameters": {"type": "object", "properties": properties, "required": list(params)},
        "handler": handler or default_handler,
    }


_ALL_TOOLS = [
    make_tool("slack_send", "슬랙 채널에 새 메시지를 보낼 때 사용. 알림, 공지, 결과 보고 전송.",
              {"channel": ("string", "채널 이름. 예: #general"), "text": ("string", "보낼 메시지 내용")},
              handler=lambda args: f"[가짜 실행 결과] {args['channel']} 채널에 메시지 전송 완료: \"{args['text']}\""),
    make_tool("slack_read", "슬랙 채널의 최근 메시지를 읽을 때 사용.",
              {"channel": ("string", "채널 이름"), "limit": ("integer", "가져올 메시지 개수")}),
    make_tool("slack_search", "슬랙 전체에서 키워드로 과거 메시지를 찾을 때 사용.",
              {"keyword": ("string", "검색 키워드")}),
    make_tool("calendar_create_event", "캘린더에 새 일정을 등록할 때 사용. 회의, 약속 생성.",
              {"title": ("string", "일정 제목"), "date": ("string", "날짜 YYYY-MM-DD"), "time": ("string", "시작 시각 HH:MM")},
              handler=lambda args: f"[가짜 실행 결과] 일정 등록 완료: {args['date']} {args['time']} \"{args['title']}\" (event_id=evt_1042)"),
    make_tool("calendar_list_events", "특정 날짜에 어떤 일정이 있는지 확인할 때 사용.",
              {"date": ("string", "날짜 YYYY-MM-DD")}),
    make_tool("calendar_delete_event", "등록된 일정을 취소할 때 사용.",
              {"event_id": ("string", "일정 ID")}),
    make_tool("gmail_send", "이메일을 보낼 때 사용.",
              {"to": ("string", "받는 사람 주소"), "subject": ("string", "제목"), "body": ("string", "본문")},
              hint="메일 이메일 전송 발송"),
    make_tool("gmail_search", "받은 메일함에서 메일을 찾을 때 사용.",
              {"query": ("string", "검색어")}),
    make_tool("gmail_read", "메일 한 통의 본문을 읽을 때 사용.",
              {"message_id": ("string", "메일 ID")}),
    make_tool("jira_create_issue", "지라에 새 이슈(티켓)를 만들 때 사용.",
              {"project": ("string", "프로젝트 키"), "title": ("string", "이슈 제목"), "description": ("string", "이슈 내용")},
              hint="지라 이슈 티켓 생성"),
    make_tool("jira_search_issues", "지라에서 이슈를 검색할 때 사용.",
              {"keyword": ("string", "검색 키워드")}),
    make_tool("jira_add_comment", "지라 이슈에 댓글을 달 때 사용.",
              {"issue_id": ("string", "이슈 ID"), "comment": ("string", "댓글 내용")}),
    make_tool("github_create_pr", "깃허브에 풀리퀘스트를 만들 때 사용.",
              {"repo": ("string", "저장소 이름"), "title": ("string", "PR 제목"), "branch": ("string", "브랜치 이름")}),
    make_tool("github_list_issues", "깃허브 저장소의 이슈 목록을 볼 때 사용.",
              {"repo": ("string", "저장소 이름")}),
    make_tool("github_merge_pr", "깃허브 풀리퀘스트를 머지할 때 사용.",
              {"repo": ("string", "저장소 이름"), "pr_number": ("integer", "PR 번호")}),
    make_tool("notion_create_page", "노션에 새 문서 페이지를 만들 때 사용.",
              {"title": ("string", "페이지 제목"), "content": ("string", "페이지 내용")}),
    make_tool("notion_search", "노션에서 문서를 검색할 때 사용.",
              {"keyword": ("string", "검색 키워드")},
              hint="노션 문서 페이지 검색"),
    make_tool("weather_get", "특정 도시의 현재 날씨를 확인할 때 사용.",
              {"city": ("string", "도시 이름. 예: 서울")},
              handler=lambda args: f"[가짜 실행 결과] {args['city']} 현재 날씨: 맑음, 기온 31도, 습도 62%",
              hint="날씨 기온 조회"),
    make_tool("translate_text", "문장을 다른 언어로 번역할 때 사용.",
              {"text": ("string", "번역할 문장"), "target_lang": ("string", "목표 언어. 예: en, ja")},
              hint="번역 언어 변환"),
    make_tool("currency_convert", "환율 기준으로 금액을 다른 통화로 바꿀 때 사용.",
              {"amount": ("number", "금액"), "from_currency": ("string", "원래 통화. 예: KRW"), "to_currency": ("string", "바꿀 통화. 예: USD")}),
    make_tool("db_query", "사내 데이터베이스에 SQL 조회를 실행할 때 사용.",
              {"sql": ("string", "실행할 SQL")},
              hint="데이터베이스 DB SQL 조회"),
    make_tool("file_read", "파일 내용을 읽을 때 사용.",
              {"path": ("string", "파일 경로")}),
    make_tool("file_write", "파일에 내용을 저장할 때 사용.",
              {"path": ("string", "파일 경로"), "content": ("string", "저장할 내용")}),
    make_tool("reminder_create", "지정한 시각에 알림을 만들 때 사용.",
              {"text": ("string", "알림 내용"), "when": ("string", "알림 시각 YYYY-MM-DD HH:MM")}),
]

REGISTRY = {t["name"]: t for t in _ALL_TOOLS}
print(f"레지스트리 도구 수: {len(REGISTRY)}개")
print(", ".join(REGISTRY))

레지스트리 도구 수: 24개
slack_send, slack_read, slack_search, calendar_create_event, calendar_list_events, calendar_delete_event, gmail_send, gmail_search, gmail_read, jira_create_issue, jira_search_issues, jira_add_comment, github_create_pr, github_list_issues, github_merge_pr, notion_create_page, notion_search, weather_get, translate_text, currency_convert, db_query, file_read, file_write, reminder_create


## 2. API에 선언하는 도구는 딱 2개

`tool_search`와 `tool_invoke`만 `tools`에 넣습니다. **이 배열은 세션 내내 바꾸지 않습니다.**

한 가지 짚을 점: `tool_invoke`의 `arguments`는 어떤 도구가 올지 몰라서 자유 형식 객체입니다.
그래서 서버가 해주는 엄격한(strict) 인자 검증은 못 씁니다. 검증은 4번 섹션에서 클라이언트가 직접 합니다.
이게 이 설계가 지불하는 대가 중 하나입니다.

In [3]:
TOOL_SEARCH_DEF = {
    "type": "function",
    "name": "tool_search",
    "description": ("사용 가능한 도구를 검색해 스키마를 로드한다. "
                    "일부 도구는 tools에 미리 선언되지 않는다(deferred) — 스키마를 로드하기 전에는 "
                    "파라미터를 모르므로 실행할 수 없고, 먼저 이 도구로 스키마를 받은 다음에만 "
                    "tool_invoke로 실행할 수 있다. 작업에 필요한 도구가 tools에 보이지 않으면 "
                    "반드시 이 도구로 먼저 검색한다. "
                    "쿼리 형식: (1) 'select:이름1,이름2' — 정확한 이름 직조회, 쉼표로 여러 개. "
                    "(2) 일반 키워드 — 조사를 뺀 명사를 공백으로 구분해 후보 검색. 예: '슬랙 메시지 전송'. "
                    "(3) '+키워드' — 반드시 매칭돼야 하는 필수 키워드."),
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": ("정확한 이름을 알면 'select:도구이름' (쉼표로 여러 개 가능). "
                                "모르면 조사를 뺀 명사 키워드를 공백으로 구분해 입력. 예: '슬랙 메시지 전송'"),
            }
        },
        "required": ["query"],
    },
    "strict": False,
}

TOOL_INVOKE_DEF = {
    "type": "function",
    "name": "tool_invoke",
    "description": ("tool_search로 스키마를 확인한 도구를 실제로 실행한다. 모든 도구 실행은 이 통로로만 한다. "
                    "스키마를 로드하지 않은 도구를 호출하면 에러와 함께 로드 방법이 안내된다."),
    "parameters": {
        "type": "object",
        "properties": {
            "name": {"type": "string", "description": "실행할 도구 이름"},
            "arguments": {"type": "object", "description": "그 도구의 스키마에 맞춘 인자 객체"},
        },
        "required": ["name", "arguments"],
    },
    "strict": False,
}

FROZEN_TOOLS = [TOOL_SEARCH_DEF, TOOL_INVOKE_DEF]  # 절대 변경하지 않는 배열

## 3. tool_search 구현 — 입력 형태에 따른 동작

| 입력 | 동작 |
|---|---|
| `select:slack_send` | 재검색 없이 레지스트리에서 바로 찾아 **풀 스키마** 리턴 (쉼표로 여러 개 가능, 없는 이름은 제외하고 알림) |
| `slack_send` (이름 그대로) | 검색 생략, 즉시 그 도구의 풀 스키마 리턴 (fast path) |
| `슬랙 메시지 전송` | 키워드 점수를 매겨 **상위 5개를 이름+설명 카드로만** 리턴. 모델이 골라서 `select:`로 다시 호출 |
| `+슬랙 메시지` | `+`가 붙은 키워드는 필수 — 그 키워드가 있는 도구만 후보에 남김 |

카드에 스키마를 안 싣는 이유: 후보 5개의 풀 스키마는 세션 끝까지 대화에 남아 토큰을 차지합니다.
필요한 것만 골라 받는 쪽이 쌉니다.

점수 규칙은 클로드코드 실제 구현(`src/tools/ToolSearchTool/ToolSearchTool.ts:186-302`)을 이식했습니다.
도구 이름을 단어로 분해한 뒤 (slack_send → slack, send) 키워드마다 점수를 더합니다.

| 매치 위치 | 점수 |
|---|---|
| 이름 단어와 정확히 일치 | +10 |
| 이름 단어에 부분 포함 | +5 |
| (여태 0점일 때) 이름 전체 문자열에 포함 | +3 |
| search_hint (도구별 큐레이션 검색 힌트) | +4 |
| 설명(description) | +2 |

클로드코드와 다르게 둔 부분 두 가지:
- 설명 매치에 클로드코드는 단어 경계 정규식(`\b`)을 쓰지만, 한글은 조사가 붙어서("메시지를") `\b`가 안 맞습니다.
  그래서 한글이 든 키워드만 substring 매칭으로 처리합니다.
- 후보 카드 단계와 "1위 점수가 2위의 2배 이상이면 바로 스키마" 숏컷은 클로드코드에 없습니다.
  클로드코드는 검색 결과를 tool_reference 블록으로 리턴하면 Anthropic API가 서버쪽에서 풀 스키마로
  확장해 주기 때문에 그런 단계가 필요 없습니다. 그 서버 확장이 없는 OpenAI라서 이 프로토콜을 씁니다.

In [4]:
TOP_N = 5
DOMINANT_RATIO = 2.0  # 1위 점수가 2위의 2배 이상이면 바로 스키마 리턴


def name_parts(name):
    # 클로드코드 parseToolName 이식: snake_case와 CamelCase를 단어로 분해
    spaced = re.sub(r"([a-z])([A-Z])", r"\1 \2", name).replace("_", " ")
    return [p for p in spaced.lower().split() if p]


def term_matches_text(term, text):
    # 클로드코드는 단어 경계 정규식(\b)을 쓰지만 한글 조사("메시지를")에는 안 맞아서
    # 한글이 든 키워드는 substring으로 매칭한다
    if re.search(r"[가-힣]", term):
        return term in text
    return re.search(r"\b" + re.escape(term) + r"\b", text) is not None


def score_tool(terms, tool):
    # 클로드코드 searchToolsWithKeywords 스코어링 이식 (가중치 10/5/3/4/2)
    parts = name_parts(tool["name"])
    full = " ".join(parts)
    desc = tool["description"].lower()
    hint = tool.get("search_hint", "").lower()
    score = 0
    for term in terms:
        if term in parts:
            score += 10          # 이름 단어 정확 일치
        elif any(term in p for p in parts):
            score += 5           # 이름 단어 부분 포함
        if score == 0 and term in full:
            score += 3           # 이름 전체 문자열 (보조)
        if hint and term_matches_text(term, hint):
            score += 4           # 큐레이션 검색 힌트
        if term_matches_text(term, desc):
            score += 2           # 설명
    return score


def tool_has_term(tool, term):
    parts = name_parts(tool["name"])
    return (term in parts or any(term in p for p in parts)
            or term_matches_text(term, tool["description"].lower())
            or (tool.get("search_hint", "") and term_matches_text(term, tool["search_hint"].lower())))


def full_schema_text(names):
    blocks = [
        json.dumps(
            {"name": REGISTRY[n]["name"],
             "description": REGISTRY[n]["description"],
             "parameters": REGISTRY[n]["parameters"]},
            ensure_ascii=False, indent=2)
        for n in names
    ]
    return ("도구 스키마:\n" + "\n".join(blocks)
            + "\n\n이제 tool_invoke(name=도구이름, arguments=스키마에 맞는 인자 객체)로 실행하세요.")


def mount_tools(names, sess):
    """실험 B 전용: 찾은 도구를 tools 배열에 추가한다. 이게 바로 캐시를 깨는 동작."""
    for n in names:
        if all(t.get("name") != n for t in sess["tools"]):
            reg = REGISTRY[n]
            sess["tools"].append({"type": "function", "name": reg["name"],
                                  "description": reg["description"],
                                  "parameters": reg["parameters"], "strict": False})
    print(f"    📌 tools 배열 변경됨: {len(sess['tools'])}개 — 다음 요청부터 프리픽스가 달라진다")
    return f"도구 장착 완료: {', '.join(names)}. 이제 이 이름으로 직접 호출하세요."


def log_search_event(sess, summary):
    # 검색이 무엇을 찾았는지 요청별 로그(search_result 열)에 기록
    if sess is not None:
        sess.setdefault("search_events", []).append(summary)


def deliver_schemas(names, sess, prefix_note=""):
    """스키마를 대화로 흘리면서 세션의 발견 집합(discovered)에 기록 — tool_invoke의 로드 게이트가 본다."""
    if sess is not None:
        sess.setdefault("discovered", set()).update(names)
    log_search_event(sess, "스키마:" + ",".join(names))
    return prefix_note + full_schema_text(names)


def handle_tool_search(query, sess=None):
    query = query.strip()
    mount_mode = bool(sess and sess.get("mount"))

    # 모드 A — select: 정확한 이름 조회 (클로드코드처럼 부분 성공 허용)
    if query.lower().startswith("select:"):
        names = [n.strip() for n in query[len("select:"):].split(",") if n.strip()]
        found = [n for n in names if n in REGISTRY]
        missing = [n for n in names if n not in REGISTRY]
        if not found:
            log_search_event(sess, "결과없음")
            return f"ERROR: 없는 도구 이름 {missing}. 키워드로 다시 검색하세요."
        note = f"\n\n(없는 이름이라 제외됨: {', '.join(missing)})" if missing else ""
        if mount_mode:
            log_search_event(sess, "장착:" + ",".join(found))
            return mount_tools(found, sess) + note
        return deliver_schemas(found, sess) + note

    q = query.lower()

    # fast path — 쿼리 전체가 도구 이름과 정확히 일치하면 즉시 스키마 (클로드코드 이식)
    if q in REGISTRY:
        if mount_mode:
            log_search_event(sess, "장착:" + q)
            return mount_tools([q], sess)
        return deliver_schemas([q], sess)

    # 모드 B — 키워드 검색. "+키워드"는 필수 조건 (클로드코드 이식)
    raw_terms = [t for t in re.split(r"\s+", q) if t]
    required = [t[1:] for t in raw_terms if t.startswith("+") and len(t) > 1]
    optional = [t for t in raw_terms if not t.startswith("+")]
    terms = required + optional if required else raw_terms

    candidates = list(REGISTRY.values())
    if required:
        candidates = [t for t in candidates if all(tool_has_term(t, r) for r in required)]

    scored = sorted(((score_tool(terms, t), t) for t in candidates), key=lambda x: -x[0])
    scored = [(s, t) for s, t in scored if s > 0]
    if not scored:
        log_search_event(sess, "결과없음")
        return "검색 결과 없음. 조사를 뺀 다른 키워드로 다시 검색하세요. 예: '메일 전송', '일정 등록'"

    top = scored[:TOP_N]
    # 숏컷 — 압도적 1위면 고르기 생략
    if len(top) == 1 or top[0][0] >= DOMINANT_RATIO * top[1][0]:
        name = top[0][1]["name"]
        if mount_mode:
            log_search_event(sess, "장착:" + name)
            return mount_tools([name], sess)
        return deliver_schemas([name], sess, "1위 점수가 압도적이라 바로 스키마를 리턴합니다.\n\n")

    log_search_event(sess, "후보:" + ",".join(t["name"] for _, t in top))
    cards = "\n".join(f"{i + 1}. {t['name']} — {t['description']} (점수 {s})"
                      for i, (s, t) in enumerate(top))
    return ("후보 도구 목록 (점수순):\n" + cards
            + "\n\n필요한 도구를 모두 골라 tool_search(query=\"select:이름1,이름2\")로 다시 호출하세요."
            + "\n맞는 것이 없으면 다른 키워드로 재검색하세요.")

## 4. tool_invoke 구현 — 검증하고 실행

`tool_invoke`는 이름으로 레지스트리에서 도구를 찾아 실행하는 통로입니다.
실행 전에 인자를 클라이언트가 직접 검증합니다 (필수 인자 누락, 타입 오류, 스키마에 없는 인자).

검증에 실패하면 예외를 던지지 않고 **`ERROR:`로 시작하는 문자열을 결과로 돌려줍니다.**
모델이 그 에러를 읽고 인자를 고쳐서 다시 호출하게 만드는 방식입니다.
스키마를 조회하지 않고 기억만으로 호출한 경우도 여기서 걸러집니다.

In [5]:
TYPE_CHECK = {"string": str, "integer": int, "number": (int, float),
              "boolean": bool, "array": list, "object": dict}


def validate_args(parameters, args):
    errors = []
    props = parameters.get("properties", {})
    for required in parameters.get("required", []):
        if required not in args:
            errors.append(f"필수 인자 '{required}' 누락")
    for key, value in args.items():
        if key not in props:
            errors.append(f"스키마에 없는 인자 '{key}'")
        elif not isinstance(value, TYPE_CHECK.get(props[key]["type"], object)):
            errors.append(f"'{key}'는 {props[key]['type']} 타입이어야 함")
    return errors


def handle_tool_invoke(name, arguments, sess=None):
    tool = REGISTRY.get(name)
    if tool is None:
        return f"ERROR: '{name}' 도구는 없습니다. tool_search로 먼저 조회하세요."
    if (sess is not None and name not in sess.get("discovered", set())
            and not any(t.get("name") == name for t in sess["tools"])):  # 장착(실험 B) 도구는 게이트 면제
        # 클로드코드 buildSchemaNotSentHint(toolExecution.ts:578-598) 이식 — 프롬프트 규칙 대신 반응형 힌트
        return (f"ERROR: '{name}'의 스키마가 아직 로드되지 않았습니다. "
                f"먼저 tool_search(query=\"select:{name}\")로 스키마를 로드한 뒤 이 호출을 다시 시도하세요.")
    if isinstance(arguments, str):  # 모델이 객체 대신 JSON 문자열로 보낸 경우
        try:
            arguments = json.loads(arguments)
        except json.JSONDecodeError:
            return "ERROR: arguments가 올바른 JSON 객체가 아닙니다. 스키마에 맞춰 다시 호출하세요."
    errors = validate_args(tool["parameters"], arguments)
    if errors:
        return "ERROR: 인자 검증 실패 — " + "; ".join(errors) + ". 스키마에 맞춰 다시 호출하세요."
    return tool["handler"](arguments)

## 5. API 호출 없이 동작 먼저 확인

검색과 실행이 의도대로 도는지 순수 함수 수준에서 확인합니다.

In [6]:
print("─── 키워드 검색 (후보가 여럿 → 카드 목록) ───")
print(handle_tool_search("슬랙 메시지 전송"))

print()
print("─── 키워드 검색 (압도적 1위 → 바로 스키마) ───")
print(handle_tool_search("날씨")[:300], "...")

print()
print("─── select: 직접 조회 ───")
print(handle_tool_search("select:slack_send")[:300], "...")

print()
print("─── fast path: 이름 그대로 입력 → 즉시 스키마 ───")
print(handle_tool_search("slack_send")[:200], "...")

print()
print("─── +필수 연산자: '메일'이 있는 도구만 후보 ───")
print(handle_tool_search("+메일 전송")[:300], "...")

print()
print("─── select: 부분 성공 — 없는 이름은 제외하고 알림 ───")
print(handle_tool_search("select:slack_send,slack_broadcast")[-120:])

─── 키워드 검색 (후보가 여럿 → 카드 목록) ───
후보 도구 목록 (점수순):
1. slack_send — 슬랙 채널에 새 메시지를 보낼 때 사용. 알림, 공지, 결과 보고 전송. (점수 6)
2. slack_read — 슬랙 채널의 최근 메시지를 읽을 때 사용. (점수 4)
3. slack_search — 슬랙 전체에서 키워드로 과거 메시지를 찾을 때 사용. (점수 4)
4. gmail_send — 이메일을 보낼 때 사용. (점수 4)

필요한 도구를 모두 골라 tool_search(query="select:이름1,이름2")로 다시 호출하세요.
맞는 것이 없으면 다른 키워드로 재검색하세요.

─── 키워드 검색 (압도적 1위 → 바로 스키마) ───
1위 점수가 압도적이라 바로 스키마를 리턴합니다.

도구 스키마:
{
  "name": "weather_get",
  "description": "특정 도시의 현재 날씨를 확인할 때 사용.",
  "parameters": {
    "type": "object",
    "properties": {
      "city": {
        "type": "string",
        "description": "도시 이름. 예: 서울"
      }
    },
    "required": [
      "city"
    ]
 ...

─── select: 직접 조회 ───
도구 스키마:
{
  "name": "slack_send",
  "description": "슬랙 채널에 새 메시지를 보낼 때 사용. 알림, 공지, 결과 보고 전송.",
  "parameters": {
    "type": "object",
    "properties": {
      "channel": {
        "type": "string",
        "description": "채널 이름. 예: #general"
      },
      "text": {
        "type": "string",
    

In [7]:
print("─── 검증 실패 (text 누락) → 에러 리턴 → 모델이 읽고 고치게 됨 ───")
print(handle_tool_invoke("slack_send", {"channel": "#deploy"}))

print()
print("─── 검증 통과 → 실행 ───")
print(handle_tool_invoke("slack_send", {"channel": "#deploy", "text": "배포 완료"}))

print()
print("─── 없는 도구 → 프로토콜로 되돌리는 에러 ───")
print(handle_tool_invoke("slack_broadcast", {}))

─── 검증 실패 (text 누락) → 에러 리턴 → 모델이 읽고 고치게 됨 ───
ERROR: 인자 검증 실패 — 필수 인자 'text' 누락. 스키마에 맞춰 다시 호출하세요.

─── 검증 통과 → 실행 ───
[가짜 실행 결과] #deploy 채널에 메시지 전송 완료: "배포 완료"

─── 없는 도구 → 프로토콜로 되돌리는 에러 ───
ERROR: 'slack_broadcast' 도구는 없습니다. tool_search로 먼저 조회하세요.


## 6. 에이전트 루프와 캐시 측정

이제 실제 API를 부릅니다. 매 요청마다 `usage`에서 두 값을 기록합니다.

- `input_tokens`: 이번 요청의 전체 입력 토큰
- `input_tokens_details.cached_tokens`: 그중 캐시에서 재사용한 토큰

확인할 때 알아둘 것:
- 캐싱은 입력이 **1024 토큰 이상**일 때부터 가능해집니다. 다만 이것은 최소 조건일 뿐입니다.
  실측(gpt-5.5, 2026-07)으로는 **읽을 수 있는 캐시가 직전 요청 프리픽스보다 200~500토큰 뒤처집니다.**
  그래서 1024를 살짝만 넘는 입력(예: 1071)은 같은 요청을 반복해도 계속 미스였습니다. (부록 1 참고)
- `cached_tokens`는 128 토큰 단위 숫자로 나옵니다. 대화가 길어지면 매 요청 계단처럼 커집니다. (부록 2 참고)
- 캐시를 쓰고 나서 1~2초 안에 온 다음 요청은 아직 그걸 못 읽을 수 있습니다. (쓰기 반영 지연)
- 캐시는 기본 24시간 유지됩니다. 그래서 **이 노트북을 다시 실행하면 첫 요청부터 HIT가 나올 수 있습니다.** 정상입니다.
- `prompt_cache_key`로 같은 프리픽스의 요청을 같은 캐시로 라우팅합니다.
- 캐시 적중은 보장이 아니라 **best-effort**입니다. 프리픽스가 완전히 같아도 미스가 날 수 있습니다.
  같은 키의 요청이 서버 여러 대로 분산되는데, 캐시는 서버마다 따로 있어서
  아직 안 데워진 서버에 떨어진 요청은 통째로 미스가 됩니다.
  이건 클라이언트에서 제어할 수 없습니다. (요청 간격을 늘리면 나아질 것 같지만,
  실측해 보면 서버를 더 자주 옮겨 다녀서 오히려 미스가 늘었습니다.)
- 그래서 이 노트북에서 볼 것은 미스의 **개수**가 아니라 **위치와 성격**입니다.
  - 서버 분산 때문에 나는 미스: 위치가 무작위이고, 실험 A/B 어느 쪽에나 똑같이 생깁니다.
  - 우리가 프리픽스를 깨서 나는 미스: 실험 B에서 tools를 바꾼 **직후에 결정적으로** 생깁니다.

In [8]:
def new_session(tools, instructions, cache_key, mount=False):
    return {"tools": list(tools), "instructions": instructions, "cache_key": cache_key,
            "mount": mount, "input_list": [], "log": [], "turn": 0,
            "search_events": [], "discovered": set()}


def record_usage(sess, response, elapsed):
    usage = response.usage
    cached = getattr(usage.input_tokens_details, "cached_tokens", 0) or 0
    cycle = sum(1 for r in sess["log"] if r["turn"] == sess["turn"]) + 1  # 턴 내 몇 번째 사이클인지
    row = {"req": len(sess["log"]) + 1, "turn": sess["turn"], "cycle": cycle, "input": usage.input_tokens,
           "cached": cached, "output": usage.output_tokens,
           "search": "-", "sec": round(elapsed, 1)}
    sess["log"].append(row)
    mark = "✅ HIT" if cached > 0 else "❌ MISS"
    print(f"    [요청 {row['req']:>2} · 사이클 {cycle}] input={row['input']:>6}  cached={cached:>6}  {mark}"
          f"  ({row['sec']}초)")


def dispatch(sess, name, args):
    if name == "tool_search":
        return handle_tool_search(args.get("query", ""), sess)
    if name == "tool_invoke":
        return handle_tool_invoke(args.get("name", ""), args.get("arguments", {}), sess)
    if name in REGISTRY:
        # tools 배열에 선언된 도구만 직접 호출 허용 (실험 B의 장착 도구)
        if any(t.get("name") == name for t in sess["tools"]):
            errors = validate_args(REGISTRY[name]["parameters"], args)
            if errors:
                return "ERROR: 인자 검증 실패 — " + "; ".join(errors)
            return REGISTRY[name]["handler"](args)
        return f"ERROR: '{name}'은 직접 호출할 수 없습니다. tool_invoke를 사용하세요."
    return f"ERROR: 알 수 없는 도구 '{name}'"


def run_turn(sess, user_msg, max_requests=8):
    sess["turn"] += 1
    print(f"\n👤 사용자: {user_msg}")
    sess["input_list"].append({"role": "user", "content": user_msg})

    for _ in range(max_requests):
        start = time.perf_counter()
        response = client.responses.create(
            model=MODEL,
            instructions=sess["instructions"],
            input=sess["input_list"],
            tools=sess["tools"],
            prompt_cache_key=sess["cache_key"],
        )
        record_usage(sess, response, time.perf_counter() - start)
        sess["input_list"] += response.output  # reasoning, function_call 등을 그대로 누적

        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:
            print(f"🤖 답변: {response.output_text}")
            return

        invoked = []
        for call in calls:
            args = json.loads(call.arguments)
            result = dispatch(sess, call.name, args)
            if call.name == "tool_invoke":
                invoked.append("실행:" + str(args.get("name", "?")))
            elif call.name != "tool_search" and call.name in REGISTRY:
                invoked.append("실행:" + call.name)
            preview = call.arguments if len(call.arguments) <= 90 else call.arguments[:90] + "…"
            print(f"    🔧 {call.name}({preview})")
            print(f"       → {result.splitlines()[0][:90]}")
            sess["input_list"].append(
                {"type": "function_call_output", "call_id": call.call_id, "output": result})

        # 이번 요청에서 검색·실행된 도구를 해당 행의 search_result로 기록
        events = sess.get("search_events", []) + invoked
        sess["search_events"] = []
        if events:
            sess["log"][-1]["search"] = " · ".join(events)

    print("⚠️ 최대 요청 횟수 도달 — 턴 종료")


def print_log(title, sess):
    print(f"═══ {title} ═══")
    print(f"{'요청':>3} {'턴':>3} {'사이클':>3} {'input':>7} {'cached':>7} {'적중률':>5} {'초':>6}  search_result")
    for r in sess["log"]:
        rate = f"{r['cached'] / r['input'] * 100:.0f}%" if r["input"] else "-"
        sr = r.get("search", "-")
        if len(sr) > 52:
            sr = sr[:52] + "…"
        print(f"{r['req']:>4} {r['turn']:>3} {r['cycle']:>5} {r['input']:>7} {r['cached']:>7} {rate:>6} {r['sec']:>6}  {sr}")
    total_input = sum(r["input"] for r in sess["log"])
    total_cached = sum(r["cached"] for r in sess["log"])
    misses = sum(1 for r in sess["log"] if r["cached"] == 0)
    print(f"합계: 요청 {len(sess['log'])}회 | 입력 {total_input:,} 토큰 | "
          f"캐시에서 재사용 {total_cached:,} 토큰 ({total_cached / total_input * 100:.0f}%) | 미스 {misses}회")

## 7. 시스템 지시문 — 도구 사용법은 여기 안 쓴다

원본 클로드코드의 시스템 프롬프트에는 "ToolSearch로 먼저 검색하고 실행하라" 같은 규칙 블록이 없습니다.
그 지식은 두 곳에 살고, 이 노트북도 같은 위치에 뒀습니다:
1. **tool_search 도구의 description** — deferred 도구 개념 + 쿼리 형식 (§2)
2. **반응형 에러 힌트** — 스키마 없이 tool_invoke를 호출하면 그때 "먼저 select:로 로드하라"를 돌려준다
   (클로드코드 `buildSchemaNotSentHint`, `toolExecution.ts:578-598` 이식)

그래서 지시문에는 페르소나 한 줄과 일반 업무 정책만 남습니다. 의도적인 선택 두 가지는 그대로입니다:
- **도구 이름 목록(카탈로그)을 지시문에 넣지 않았습니다.** 넣으면 모델이 검색을 건너뛰고 바로 `select:`로 가는 경우가 많아서, 키워드 검색 과정을 보여주기 어렵습니다. 실전에서는 넣는 쪽이 왕복을 줄여줍니다.
- 지시문(정책 블록)은 일부러 넉넉하게 유지합니다. 캐싱이 1024 토큰부터 동작하므로, 고정 프리픽스(tools + 지시문)가 처음부터 그 기준을 넘게 만들기 위해서입니다.

실험 B(장착 구방식)의 지시문 RULES_V1은, 구방식 흐름 자체가 클로드코드에 없는 설계라 규칙 서술을 유지합니다.

In [9]:
PERSONA = "너는 사내 업무 비서다. 사용자의 요청을 도구를 사용해서 처리한다.\n"

COMMON_POLICY = """[작업 정책]
- 날짜는 YYYY-MM-DD 형식, 시각은 HH:MM 24시간 형식으로 도구에 넘긴다. 연도가 없으면 2026년으로 본다.
- 메시지나 메일 본문은 사용자가 준 문구를 그대로 쓰고, 내용을 마음대로 추가하지 않는다.
- 삭제, 머지, 결제처럼 되돌리기 어려운 작업은 실행하기 전에 사용자에게 한 번 확인한다.
- 개인정보는 도구 인자에 꼭 필요한 경우에만 넣는다.
- 검색 결과가 비어 있으면 키워드를 바꿔서 한 번 더 검색하고, 그래도 없으면 없다고 보고한다.
- 한 요청에 여러 작업이 있으면 순서대로 하나씩 처리한다.
- 도구 실행 결과에 오류가 있으면 그 내용을 사용자에게 그대로 알린다.
- 실행하지 않은 작업을 했다고 말하지 않는다.
- 필요한 인자가 요청에 없으면 합리적인 값을 채우되, 최종 답변에서 그 사실을 밝힌다.
- 모든 시각은 한국 표준시(Asia/Seoul) 기준으로 해석한다.
- 도구를 실행하기 전에 스키마의 required 목록에 있는 인자가 전부 채워졌는지 확인한다.
- 같은 도구를 같은 인자로 두 번 연속 호출하지 않는다.
- 사용자가 시키지 않은 도구 실행은 하지 않는다.
- 도구 이름과 인자 이름은 스키마에 적힌 그대로 쓴다. 임의로 줄이거나 바꾸지 않는다.
- 답변 첫머리에 인사말, 감탄사, 이모지를 넣지 않는다.
- 작업이 끝나면 어떤 도구를 실행했고 결과가 무엇이었는지 한 줄로 요약해서 답한다.
- 최종 답변은 한국어로 간결하게 쓴다.
"""

FORMAT_POLICY = """[응답 형식 정책]
- 도구 실행 전에 어떤 도구를 왜 쓰는지 속으로만 판단하고, 사용자에게는 결과만 보고한다.
- 여러 도구를 실행했으면 실행한 순서대로 결과를 정리한다.
- 숫자, 날짜, 채널 이름, 파일 키 같은 식별자는 도구 결과에 나온 그대로 인용한다.
- 도구 결과가 길면 사용자 질문과 관련된 부분만 추려서 전달한다.
- 같은 턴에서 이미 확인한 정보는 다시 도구를 호출하지 않고 재사용한다.
- 도구가 빈 결과를 돌려주면 결과가 없다는 사실을 그대로 보고한다.
- 추측으로 정보를 만들어내지 않는다. 도구 결과에 없는 내용은 없다고 말한다.
- 작업을 절반만 처리한 경우 어디까지 됐고 무엇이 남았는지 명확히 구분해서 알린다.
- 에러 메시지를 사용자에게 전달할 때는 원문 그대로 인용한 뒤 한 줄 설명을 덧붙인다.
- 목록을 보고할 때는 항목당 한 줄로 정리하고, 항목이 다섯 개를 넘으면 개수를 먼저 말한다.
- 사용자가 요청한 범위를 넘는 정보는 묻기 전에는 덧붙이지 않는다.
"""

SAFETY_POLICY = """[안전 정책]
- 되돌리기 어려운 작업(삭제, 머지, 대량 발송)은 실행 전에 대상과 범위를 한 번 더 확인한다.
- 외부로 나가는 메시지에는 내부 식별자나 토큰 값을 포함하지 않는다.
- 사용자가 명시하지 않은 수신자를 임의로 추가하지 않는다.
- 민감한 값(비밀번호, 키)은 출력에 마스킹해서 표시한다.
- 실패한 작업을 성공했다고 요약하지 않는다.
- 오래 걸리는 작업은 시작했다는 사실을 먼저 알린다.
- 동일 요청이 반복되면 직전 결과를 재사용할지 사용자에게 확인한다.
- 도구 결과와 사용자 기대가 상충하면 도구 결과를 기준으로 보고하고 차이를 명시한다.
"""

# 정책 블록들이 동결 프리픽스를 캐시 최소 단위(1024토큰) 위로 여유 있게 올린다
INSTRUCTIONS_V2 = PERSONA + "\n" + COMMON_POLICY + "\n" + FORMAT_POLICY + "\n" + SAFETY_POLICY
print(f"지시문 길이: {len(INSTRUCTIONS_V2)}자 — 도구 사용 규칙 없음")

지시문 길이: 1589자 — 도구 사용 규칙 없음


## 8. 실험 A — tools 고정 방식 (이 노트북의 설계)

같은 세션에서 사용자 요청 3개를 처리합니다. 도구 검색과 실행이 여러 번 일어납니다.

기대하는 패턴:
- **요청 1: MISS** (캐시가 처음 만들어지는 정상 과정. 이 노트북을 이미 실행한 적 있으면 HIT일 수도 있음)
- **요청 2부터: 대부분 HIT**, 대화가 쌓일수록 `cached_tokens`도 같이 늘어남
- tools 배열은 끝까지 2개로 고정 — 표의 search_result 열에서 무엇이 검색·실행됐는지 볼 수 있음

중간에 미스가 몇 번 섞일 수 있습니다. 위에서 말한 서버 분산 때문이고,
우리가 요청 앞부분을 바꿔서 생긴 미스가 아닙니다. 그 증거는 미스가 나온 **뒤에도**
`cached_tokens`가 계속 커진다는 것입니다. 프리픽스가 진짜 깨졌다면 그 지점 이후로 다시 작아져야 합니다.

In [10]:
sess_a = new_session(tools=FROZEN_TOOLS, instructions=INSTRUCTIONS_V2,
                     cache_key="toolsearch-v2-demo")

run_turn(sess_a, "슬랙 #deploy 채널에 '배포 완료'라고 메시지 보내줘")


👤 사용자: 슬랙 #deploy 채널에 '배포 완료'라고 메시지 보내줘


    [요청  1 · 사이클 1] input=  1247  cached=     0  ❌ MISS  (9.2초)
    🔧 tool_search({"query":"슬랙 메시지 전송"})
       → 후보 도구 목록 (점수순):


    [요청  2 · 사이클 2] input=  2292  cached=  2048  ✅ HIT  (1.7초)
    🔧 tool_invoke({"name":"slack_send","arguments":{"channel":"#deploy","text":"배포 완료"}})
       → ERROR: 'slack_send'의 스키마가 아직 로드되지 않았습니다. 먼저 tool_search(query="select:slack_send")로 스키마를 로


    [요청  3 · 사이클 3] input=  2528  cached=  2432  ✅ HIT  (1.1초)
    🔧 tool_search({"query":"select:slack_send"})
       → 도구 스키마:


    [요청  4 · 사이클 4] input=  2750  cached=  2432  ✅ HIT  (1.0초)
    🔧 tool_invoke({"name":"slack_send","arguments":{"channel":"#deploy","text":"배포 완료"}})
       → [가짜 실행 결과] #deploy 채널에 메시지 전송 완료: "배포 완료"


    [요청  5 · 사이클 5] input=  2845  cached=  2688  ✅ HIT  (8.4초)
🤖 답변: #deploy 채널에 메시지 전송 완료: "배포 완료"
요약: 실행 도구 slack_send, 결과: #deploy 채널에 메시지 전송 완료: "배포 완료"


In [11]:
run_turn(sess_a, "7월 24일 오후 3시에 '주간 회의' 일정 등록해줘")


👤 사용자: 7월 24일 오후 3시에 '주간 회의' 일정 등록해줘


    [요청  6 · 사이클 1] input=  1867  cached=  1152  ✅ HIT  (22.9초)
🤖 답변: 일정을 자동으로 등록하는 캘린더 도구에 연결되어 있지 않아 바로 등록은 할 수 없습니다.

다음과 같이 진행 가능합니다. 원하시는 방식이나 정보가 있으면 알려주세요.

- ICS 파일(수동 등록용) 예시
BEGIN:VCALENDAR
VERSION:2.0
BEGIN:VEVENT
UID:20260724T150000-001@example.com
DTSTART:20260724T150000
DTEND:20260724T160000
SUMMARY:주간 회의
END:VEVENT
END:VCALENDAR

- 간단한 일정 요약(복사해 바로 메모/일정 입력에 사용 가능)
주간 회의 | 시작: 2026-07-24 15:00 | 종료: 2026-07-24 16:00 | 캘린더: 기본 캘린더 | 장소: 미정 | 참석자: 미정

확인하고 싶은 점
- 참석자(add_attendees) 여부: 참석자를 초대하시겠습니까? (예: 팀원들 이메일)
- 장소(location): 회의 장소가 있나요?
- 소요 시간(duration): 기본 60분으로 두면 될까요, 다른 시간대가 필요하신가요?
- 캘린더(calendar): 특정 캘린더가 있다면 알려주시면 그 캘린더로 맞춰 드리겠습니다. 기본 캘린더로 등록해도 될까요?

원하시는 방식과 세부 정보를 알려주시면 바로 정리해서 드리겠습니다.


In [12]:
run_turn(sess_a, "서울 날씨 알려줘")


👤 사용자: 서울 날씨 알려줘


    [요청  7 · 사이클 1] input=  2204  cached=  1792  ✅ HIT  (8.4초)
    🔧 tool_search({"query":"select:weather"})
       → ERROR: 없는 도구 이름 ['weather']. 키워드로 다시 검색하세요.


    [요청  8 · 사이클 2] input=  3125  cached=  3072  ✅ HIT  (1.8초)
    🔧 tool_search({"query":"날씨 서울"})
       → 1위 점수가 압도적이라 바로 스키마를 리턴합니다.


    [요청  9 · 사이클 3] input=  3442  cached=  3200  ✅ HIT  (1.3초)
    🔧 tool_invoke({"name":"weather_get","arguments":{"city":"서울"}})
       → [가짜 실행 결과] 서울 현재 날씨: 맑음, 기온 31도, 습도 62%


    [요청 10 · 사이클 4] input=  3564  cached=  3456  ✅ HIT  (4.6초)
🤖 답변: 서울 현재 날씨: 맑음, 기온 31도, 습도 62%

요약: 실행 도구 weather_get, 결과: 서울 현재 날씨: 맑음, 기온 31도, 습도 62%


In [13]:
print_log("실험 A — tools 고정 (미스 0 설계)", sess_a)

═══ 실험 A — tools 고정 (미스 0 설계) ═══
 요청   턴 사이클   input  cached   적중률      초  search_result
   1   1     1    1247       0     0%    9.2  후보:slack_send,slack_read,slack_search,gmail_send
   2   1     2    2292    2048    89%    1.7  실행:slack_send
   3   1     3    2528    2432    96%    1.1  스키마:slack_send
   4   1     4    2750    2432    88%    1.0  실행:slack_send
   5   1     5    2845    2688    94%    8.4  -
   6   2     1    1867    1152    62%   22.9  -
   7   3     1    2204    1792    81%    8.4  결과없음
   8   3     2    3125    3072    98%    1.8  스키마:weather_get
   9   3     3    3442    3200    93%    1.3  실행:weather_get
  10   3     4    3564    3456    97%    4.6  -
합계: 요청 10회 | 입력 25,864 토큰 | 캐시에서 재사용 22,272 토큰 (86%) | 미스 1회


위 표에서 볼 것:

- search_result 열에 검색(후보/스키마)과 실행이 계속 찍히는데도 tools 배열은 끝까지 2개입니다.
  요청 앞부분이 한 번도 안 바뀌었다는 뜻입니다.
- `cached_tokens`가 대화가 쌓일수록 계단처럼 늘어납니다 (1024 → 1536 → 2048 ...).
  직전 요청까지의 대화가 그대로 다음 요청의 캐시가 되기 때문입니다.
  중간에 미스가 있어도 이 계단이 유지된다는 것이 프리픽스가 안 깨졌다는 증거입니다.
  (서버마다 데워진 정도가 달라서 적중량이 일시적으로 작아질 수 있지만, 더 큰 값이 다시 나온다는 것 자체가 프리픽스가 살아 있다는 뜻입니다.)
- 이 데모는 턴이 얇아서(요청당 100~200토큰) 같은 값에 여러 요청이 머물러 보입니다.
  턴이 굵으면 매 요청 cached가 커지는 것이 보입니다 — 부록 2에서 확인합니다.
- 미스는 무작위 위치에 흩어져 있습니다 (서버 분산 노이즈). 도구를 새로 발견했다고 미스가 나는 일은 없습니다.
  이게 실험 B와의 차이입니다.

## 9. 실험 B — 비교: 찾은 도구를 tools 배열에 장착하는 구방식

이번에는 설계 문서에서 v1이라고 부르는 방식입니다.
`select:` 조회가 성공하면 그 도구를 **`tools` 배열에 추가**하고, 모델이 직접 호출하게 합니다.

기대하는 패턴: **도구를 장착할 때마다, 그 다음 요청이 MISS**로 찍힙니다.
`tools`가 요청 맨 앞에 들어가므로, 배열이 바뀌면 그 뒤 전체(지시문 + 쌓인 대화)의 캐시를 전부 버리고 다시 만들기 때문입니다.
search_result 열에 '장착:'이 찍힌 직후의 행을 보세요. (장착 순간에는 📌 표시도 출력됩니다)

참고: 실제 v1 방식의 장점인 서버측 strict 인자 검증은 이 실험에서는 생략했습니다. 여기서는 캐시 차이만 봅니다.

한 가지 더: 이 노트북을 그대로 다시 실행하면 지난 실행 때 만들어진 캐시(24시간 유지)에 맞아서
두 실험 모두 미스가 훨씬 적게 보일 수 있습니다. 미스 패턴을 처음부터 다시 보고 싶으면
시나리오 문구(사용자 요청)나 시스템 지시문을 조금 바꿔서 실행하세요.

In [14]:
RULES_V1 = """너는 사내 업무 비서다. 사용자의 요청을 도구를 사용해서 처리한다.

[도구 사용 규칙 — 반드시 이 순서대로 진행한다]
1. 처음에 열려 있는 도구는 tool_search 하나뿐이다.
2. 작업에 필요한 도구는 먼저 tool_search로 찾는다.
   - 도구 이름을 정확히 알면 tool_search(query="select:도구이름")으로 조회한다.
   - 이름을 모르면 조사를 뺀 명사 키워드를 공백으로 이어서 검색한다. 예: "슬랙 메시지 전송"
3. 검색 결과로 후보 도구 목록이 오면, 필요한 도구를 골라 tool_search(query="select:...")로 다시 호출한다.
4. select: 조회가 성공하면 그 도구가 너의 도구 목록에 새로 장착된다.
   "장착 완료" 응답을 받으면, 그 도구를 이름 그대로 직접 호출해서 작업을 실행한다.
5. 도구가 ERROR로 시작하는 결과를 돌려주면, 에러 내용을 읽고 인자를 고쳐서 다시 호출한다.

[예시 흐름]
사용자: "팀에게 메일 보내줘"
1) tool_search(query="메일 전송")
2) 후보 목록 수신: gmail_send, gmail_search ...
3) tool_search(query="select:gmail_send")
4) "장착 완료" 수신 — 이제 gmail_send를 직접 쓸 수 있다
5) gmail_send를 직접 호출: to, subject, body 인자 사용
6) 실행 결과를 확인하고 사용자에게 한국어로 보고
"""

INSTRUCTIONS_V1 = RULES_V1 + "\n" + COMMON_POLICY

sess_b = new_session(tools=[TOOL_SEARCH_DEF], instructions=INSTRUCTIONS_V1,
                     cache_key="toolsearch-v1-demo", mount=True)

run_turn(sess_b, "슬랙 #deploy 채널에 '배포 완료'라고 메시지 보내줘")
run_turn(sess_b, "7월 24일 오후 3시에 '주간 회의' 일정 등록해줘")
run_turn(sess_b, "서울 날씨 알려줘")


👤 사용자: 슬랙 #deploy 채널에 '배포 완료'라고 메시지 보내줘


    [요청  1 · 사이클 1] input=  1031  cached=     0  ❌ MISS  (3.9초)
    🔧 tool_search({"query":"슬랙 메시지 전송"})
       → 후보 도구 목록 (점수순):


    [요청  2 · 사이클 2] input=  1732  cached=  1536  ✅ HIT  (1.9초)
    📌 tools 배열 변경됨: 2개 — 다음 요청부터 프리픽스가 달라진다
    🔧 tool_search({"query":"select:slack_send"})
       → 도구 장착 완료: slack_send. 이제 이 이름으로 직접 호출하세요.


    [요청  3 · 사이클 3] input=  2035  cached=     0  ❌ MISS  (2.6초)
    🔧 slack_send({"channel":"#deploy","text":"배포 완료"})
       → [가짜 실행 결과] #deploy 채널에 메시지 전송 완료: "배포 완료"


    [요청  4 · 사이클 4] input=  2361  cached=  2304  ✅ HIT  (6.8초)
🤖 답변: 도구 slack_send로 Slack 채널 #deploy에 '배포 완료'를 전송했습니다. 결과: 성공.

👤 사용자: 7월 24일 오후 3시에 '주간 회의' 일정 등록해줘


...

    [요청  7 · 사이클 3] input=  2550  cached=     0  ❌ MISS  (2.7초)
    🔧 calendar_create_event({"title":"주간 회의","date":"2026-07-24","time":"15:00"})
       → [가짜 실행 결과] 일정 등록 완료: 2026-07-24 15:00 "주간 회의" (event_id=evt_1042)


    [요청  8 · 사이클 4] input=  2867  cached=  2688  ✅ HIT  (9.6초)
🤖 답변: calendar_create_event 실행: 일정 등록 완료: 2026-07-24 15:00 '주간 회의' (이벤트 ID: evt_1042).

👤 사용자: 서울 날씨 알려줘


    [요청  9 · 사이클 1] input=  1741  cached=     0  ❌ MISS  (2.9초)
    📌 tools 배열 변경됨: 4개 — 다음 요청부터 프리픽스가 달라진다
    🔧 tool_search({"query":"서울 날씨"})
       → 도구 장착 완료: weather_get. 이제 이 이름으로 직접 호출하세요.


    [요청 10 · 사이클 2] input=  2073  cached=     0  ❌ MISS  (1.2초)
    🔧 weather_get({"city":"서울"})
       → [가짜 실행 결과] 서울 현재 날씨: 맑음, 기온 31도, 습도 62%


    [요청 11 · 사이클 3] input=  2129  cached=  2048  ✅ HIT  (1.3초)
🤖 답변: 서울 현재 날씨: 맑음, 기온 31도, 습도 62%


In [15]:
print_log("실험 B — 발견한 도구를 tools에 장착 (구방식)", sess_b)

═══ 실험 B — 발견한 도구를 tools에 장착 (구방식) ═══
 요청   턴 사이클   input  cached   적중률      초  search_result
   1   1     1    1031       0     0%    3.9  후보:slack_send,slack_read,slack_search,gmail_send
   2   1     2    1732    1536    89%    1.9  장착:slack_send
   3   1     3    2035       0     0%    2.6  실행:slack_send
   4   1     4    2361    2304    98%    6.8  -
   5   2     1    1444    1024    71%    6.2  결과없음
   6   2     2    2385    2304    97%    1.5  장착:calendar_create_event
   7   2     3    2550       0     0%    2.7  실행:calendar_create_event
   8   2     4    2867    2688    94%    9.6  -
   9   3     1    1741       0     0%    2.9  장착:weather_get
  10   3     2    2073       0     0%    1.2  실행:weather_get
  11   3     3    2129    2048    96%    1.3  -
합계: 요청 11회 | 입력 22,348 토큰 | 캐시에서 재사용 11,904 토큰 (53%) | 미스 5회


## 10. 두 방식 비교

In [16]:
def summarize(label, sess):
    log = sess["log"]
    total_input = sum(r["input"] for r in log)
    total_cached = sum(r["cached"] for r in log)
    misses = sum(1 for r in log if r["cached"] == 0)
    mounts = [i for i, r in enumerate(log) if "장착:" in r.get("search", "")]
    structural = sum(1 for i in mounts if i + 1 < len(log) and log[i + 1]["cached"] == 0)
    if mounts:
        extra = f" | 장착 {len(mounts)}회 → 장착 직후 미스 {structural}회"
    else:
        extra = " | 장착 0회 → 구조적 미스 0회"
    print(f"{label:<32} 요청 {len(log):>2}회 | 미스 {misses}회 | "
          f"캐시 재사용 {total_cached:>7,} / {total_input:>7,} 토큰 "
          f"({total_cached / total_input * 100:.0f}%){extra}")


summarize("실험 A — tools 고정", sess_a)
summarize("실험 B — 발견 시 장착", sess_b)

실험 A — tools 고정                  요청 10회 | 미스 1회 | 캐시 재사용  22,272 /  25,864 토큰 (86%) | 장착 0회 → 구조적 미스 0회
실험 B — 발견 시 장착                   요청 11회 | 미스 5회 | 캐시 재사용  11,904 /  22,348 토큰 (53%) | 장착 3회 → 장착 직후 미스 3회


숫자는 실행마다 다릅니다. 12요청짜리 작은 실험은 무작위 미스 몇 개에 합계가 크게 흔들려서,
실행에 따라 두 실험의 재사용률이 비슷해 보이거나 심지어 역전될 수도 있습니다.
서버 분산 노이즈는 양쪽에 똑같이 끼기 때문입니다.

그래서 노이즈에 흔들리지 않는 지표는 위 요약의 마지막 항목, **장착 직후 요청의 미스**입니다.

- **실험 A**: 장착이 없으므로 이 미스가 구조적으로 0입니다. 나오는 미스는 전부 무작위 위치의 서버 노이즈이고,
  미스가 있어도 `cached_tokens` 계단은 계속 올라갑니다.
- **실험 B**: 도구를 장착할 때마다 **그 직후 요청이 결정적으로** 미스입니다. search_result 열의 '장착:' 다음 행을 확인하세요.
  이때는 이미 쌓인 대화 전체를 다시 캐시에 써야 합니다. 대화가 길수록, 도구 발견이 잦을수록 이 비용은 커집니다.

안정적인 큰 표본은 부록 3입니다 — 44요청 세션에서 재사용률이 실행마다 79~83%로 일정하게 나옵니다.

## 부록 1 — 실험 A에 남는 미스는 어디서 오나: 동일 요청 반복 실험

실험 A는 프리픽스를 안 깨는데도 미스가 몇 개 남습니다. 원인을 분리하는 방법은 단순합니다.
대화를 쌓지 않고, tools도 그대로 두고, **바이트 단위로 완전히 똑같은 요청을 12번** 보냅니다.
프리픽스가 바뀔 여지가 전혀 없으므로, 여기서 나는 미스는 전부 우리 설계 밖(서버쪽)의 원인입니다.

아래 셀은 이 실험을 지시문 길이만 다르게 두 번 돌립니다. 결과가 완전히 다릅니다.

1. **짧은 버전** — 입력이 1024를 살짝 넘는 수준: 전부 또는 대부분 미스가 나옵니다.
   읽을 수 있는 캐시가 프리픽스보다 수백 토큰 뒤처지는 탓에, 이 구간은 캐시가 아예 안 잡힙니다.
2. **넉넉한 버전** — 입력이 1300 토큰대: 최초 생성 1회 + 쓰기 반영 지연 1회 + 무작위 소수를 빼고 적중합니다.

이 노트북을 다시 실행하면 지난 실행의 캐시 때문에 첫 시도부터 HIT일 수 있습니다.

In [17]:
EXTRA_POLICY = """
[보고 형식]
- 도구 실행이 성공한 경우: "완료"로 시작하고, 실행한 도구 이름과 핵심 결과를 이어서 적는다.
- 도구 실행이 실패한 경우: "실패"로 시작하고, 실패한 도구 이름과 에러 내용을 이어서 적는다.
- 도구를 찾지 못한 경우: "불가"로 시작하고, 어떤 키워드로 검색했는지 적는다.
- 확인이 필요한 경우: "확인 필요"로 시작하고, 무엇을 확인해야 하는지 적는다.
- 보고는 세 문장을 넘기지 않는다. 표나 목록은 사용자가 요청할 때만 쓴다.

[금지 사항]
- 도구 실행 결과에 없는 내용을 지어내서 보고하지 않는다.
- 같은 검색어로 tool_search를 두 번 연속 호출하지 않는다.
- 스키마를 받지 않은 도구를 tool_invoke로 실행하지 않는다.
- 사용자의 요청 범위를 벗어난 도구를 실행하지 않는다.
- 도구 실행 결과를 요약하면서 숫자나 이름을 바꾸지 않는다.
- 에러 메시지를 사용자에게 숨기지 않는다.
"""

CTRL_SHORT = "[부록: 동일 요청 반복 실험 전용 버전]\n" + INSTRUCTIONS_V2
CTRL_LONG = CTRL_SHORT + EXTRA_POLICY
CTRL_INPUT = [{"role": "user", "content": "서울 날씨 알려줘"}]


def repeat_identical(label, instructions, cache_key, n=12):
    print(f"── {label} ──")
    hits = 0
    for i in range(1, n + 1):
        start = time.perf_counter()
        r = client.responses.create(model=MODEL, instructions=instructions, input=CTRL_INPUT,
                                    tools=FROZEN_TOOLS, prompt_cache_key=cache_key)
        elapsed = time.perf_counter() - start
        cached = getattr(r.usage.input_tokens_details, "cached_tokens", 0) or 0
        hits += cached > 0
        print(f"[시도 {i:>2}] input={r.usage.input_tokens:>5}  cached={cached:>5}  "
              f"{'✅ HIT' if cached else '❌ MISS'}  ({elapsed:.1f}초)")
        time.sleep(1)
    print(f"동일 요청 {n}회: HIT {hits} / MISS {n - hits}\n")


repeat_identical("짧은 버전 (1024를 살짝 넘는 입력)", CTRL_SHORT, "identical-repeat-short")
repeat_identical("넉넉한 버전 (정책 추가로 입력 확대)", CTRL_LONG, "identical-repeat-long")

── 짧은 버전 (1024를 살짝 넘는 입력) ──


[시도  1] input= 1249  cached=    0  ❌ MISS  (4.1초)


[시도  2] input= 1249  cached=    0  ❌ MISS  (3.1초)


[시도  3] input= 1249  cached= 1152  ✅ HIT  (2.7초)


[시도  4] input= 1249  cached= 1152  ✅ HIT  (2.9초)


...

[시도  9] input= 1499  cached=    0  ❌ MISS  (4.2초)


[시도 10] input= 1499  cached= 1408  ✅ HIT  (2.1초)


[시도 11] input= 1499  cached= 1408  ✅ HIT  (2.4초)


[시도 12] input= 1499  cached= 1408  ✅ HIT  (2.6초)


동일 요청 12회: HIT 8 / MISS 4



넉넉한 버전에서 미스가 나는 자리를 보면 실험 A의 잔여 미스가 전부 설명됩니다.

- **시도 1** — 캐시 최초 생성. 어떤 설계로도 피할 수 없습니다.
- **초반 시도의 미스** — 쓰기 반영 지연. 캐시를 쓴 직후 1~2초 안의 요청은 아직 못 읽을 수 있습니다.
  실험 A/B에서 매 세션의 요청 2가 자주 미스였던 이유입니다. (반영이 빨라서 바로 적중하는 실행도 있습니다)
- **중간의 산발 미스** — 서버 분산. 같은 키의 요청이 서버 여러 대에 나뉘고 캐시는 서버마다 따로라,
  아직 안 데워진 서버에 떨어지면 통째로 미스입니다.

셋 다 프리픽스와 무관하고, 클라이언트 설계로 제거할 수 없습니다.
설계로 제거할 수 있는 것은 실험 B가 보여준 구조적 미스(tools 변경에 의한 프리픽스 파괴)뿐이며,
그것을 0으로 만드는 것이 이 노트북의 설계입니다.

## 부록 2 — 멀티턴에서 캐시는 계속 쌓인다 (턴이 굵을 때)

실험 A의 표만 보면 cached가 1024, 1536 같은 값에 머물러서 "캐시 갱신이 안 되나?" 싶을 수 있습니다.
갱신은 매 요청 자동으로 일어나고 있습니다. OpenAI 캐싱에는 Anthropic의 cache_control처럼
클라이언트가 캐시 지점을 지정·갱신하는 수단이 아예 없고, **전체 대화를 보내는 것 자체가 곧 쓰기**입니다.

머물러 보이는 이유는 두 가지입니다.

1. 읽을 수 있는 캐시는 직전 요청 프리픽스보다 200~500토큰 뒤처집니다.
2. 실험 A의 턴은 요청당 100~200토큰씩만 자라서, 여러 요청이 같은 값을 읽습니다.

턴을 굵게(턴당 약 1,300토큰) 만들면 매 요청 cached가 커지는 것이 바로 보입니다.
두 가지 버전으로 확인합니다.

- **2-1. 순수 대화 버전** — 도구 없이 회의록 요약만. 캐시 누적 규칙 자체를 다른 요인 없이 봅니다.
- **2-2. ToolSearch 세션 버전** — 실험 A와 완전히 같은 구성(tools 2개 동결 + 검색/실행 프로토콜)에
  굵은 턴을 넣은 실전형. 도구 검색과 실행이 돌면서도 캐시가 똑같이 쌓이는 것을 봅니다.

### 부록 2-1. 순수 대화 버전 (도구 없음)

In [18]:
GROWTH_INSTRUCTIONS = """[멀티턴 캐시 누적 실험 전용 지시문]
너는 문서 요약 비서다. 사용자가 주는 회의록을 요약한다.

[요약 규칙]
- 요약은 세 문장 이내로 쓴다.
- 회의록에 없는 내용을 지어내지 않는다.
- 숫자와 이름은 원문 그대로 옮긴다.
- 결정 사항과 미결 사항을 구분해서 적는다.
- 최종 답변은 한국어로 간결하게 쓴다.
- 답변 첫머리에 인사말이나 감탄사를 넣지 않는다.
- 회의 참석자 수, 날짜, 장소가 있으면 요약 첫 문장에 포함한다.
- 예산이나 금액이 나오면 반드시 요약에 포함한다.
- 담당자 배정이 나오면 담당자 이름과 업무를 함께 적는다.
- 다음 회의 일정이 문서에 있으면 요약 마지막에 적는다.
"""


def fake_minutes(n):
    lines = [f"[{n}차 주간회의 회의록 — 2026-07-{10 + n:02d}, 참석 7명, 3회의실]"]
    for i in range(1, 11):
        lines.append(
            f"안건 {i}: {n}차 스프린트 항목 {i}번에 대해 논의했다. "
            f"담당자 김개발이 진행 상황을 공유했고, 예상 완료일은 2026-08-{i:02d}로 잡았다. "
            f"품질 검증은 박테스트가 맡기로 했으며, 예산은 {i * 100}만 원 한도로 승인됐다. "
            f"미결 사항으로는 외부 업체 견적 비교가 남아 있고 다음 회의까지 자료를 준비하기로 했다."
        )
    lines.append(f"다음 회의: 2026-07-{17 + n:02d} 오전 10시.")
    return "\n".join(lines)


growth_input = []
prev_input = 0
print(f"{'요청':>3} {'input':>7} {'cached':>7} {'직전input':>9}  판정")
for turn in range(1, 8):
    growth_input.append({"role": "user", "content": f"다음 회의록을 요약해줘.\n\n{fake_minutes(turn)}"})
    start = time.perf_counter()
    r = client.responses.create(
        model=MODEL,
        instructions=GROWTH_INSTRUCTIONS,
        input=growth_input,
        prompt_cache_key="multiturn-growth-demo",
    )
    elapsed = time.perf_counter() - start
    cached = getattr(r.usage.input_tokens_details, "cached_tokens", 0) or 0
    grew = "↑ 커짐" if cached > 0 else "미스"
    print(f"{turn:>4} {r.usage.input_tokens:>7} {cached:>7} {prev_input:>9}  {grew} ({elapsed:.1f}초)")
    prev_input = r.usage.input_tokens
    growth_input += r.output
    time.sleep(1)

 요청   input  cached   직전input  판정


   1    1293       0         0  미스 (22.8초)


   2    2628    1152      1293  ↑ 커짐 (11.8초)


   3    3980    2560      2628  ↑ 커짐 (10.5초)


   4    5313    3840      3980  ↑ 커짐 (19.2초)


   5    6687    5248      5313  ↑ 커짐 (7.3초)


   6    8001    6528      6687  ↑ 커짐 (13.1초)


   7    9359    7936      8001  ↑ 커짐 (10.1초)


cached가 직전 요청 input을 뒤따라 매 요청 올라가는 것이 보입니다.
값이 직전 input보다 200~500토큰 작은 것이 "뒤처짐"이고, 큰 구간에서는 2304, 3840처럼
128 단위 숫자가 그대로 나옵니다. 초반 미스 1~2회는 부록 1과 같은 최초 생성 + 쓰기 반영 지연입니다.

### 부록 2-2. ToolSearch 세션 버전 (실험 A와 같은 구성, 굵은 턴)

이번에는 tools 2개 동결 + 검색/실행 프로토콜 그대로, 매 턴 회의록을 요약해서
슬랙으로 보내라고 시킵니다. 도구 검색·실행 요청이 섞여도 캐시가 똑같이 쌓이는지 봅니다.

하나 더 볼 것: 턴 1에서 받은 발견 결과가 대화에 남아 있으므로,
턴 2부터는 모델이 키워드 검색(카드 고르기) 단계를 건너뛰고 select: 직조회나 곧장 tool_invoke로 갑니다.
발견 결과가 캐시된 컨텍스트의 일부가 되어 재사용되는 것 — 이 설계가 노리는 효과 그대로입니다.

In [19]:
sess_growth = new_session(tools=FROZEN_TOOLS, instructions=INSTRUCTIONS_V2,
                          cache_key="toolsearch-growth-demo")

for turn in range(1, 5):
    run_turn(sess_growth,
             f"다음 {turn}차 회의록을 세 문장으로 요약해서 슬랙 #minutes 채널에 보내줘.\n\n{fake_minutes(turn)}")

print()
print_log("부록 2-2 — ToolSearch 세션, 굵은 턴 멀티턴", sess_growth)


👤 사용자: 다음 1차 회의록을 세 문장으로 요약해서 슬랙 #minutes 채널에 보내줘.

[1차 주간회의 회의록 — 2026-07-11, 참석 7명, 3회의실]
안건 1: 1차 스프린트 항목 1번에 대해 논의했다. 담당자 김개발이 진행 상황을 공유했고, 예상 완료일은 2026-08-01로 잡았다. 품질 검증은 박테스트가 맡기로 했으며, 예산은 100만 원 한도로 승인됐다. 미결 사항으로는 외부 업체 견적 비교가 남아 있고 다음 회의까지 자료를 준비하기로 했다.
안건 2: 1차 스프린트 항목 2번에 대해 논의했다. 담당자 김개발이 진행 상황을 공유했고, 예상 완료일은 2026-08-02로 잡았다. 품질 검증은 박테스트가 맡기로 했으며, 예산은 200만 원 한도로 승인됐다. 미결 사항으로는 외부 업체 견적 비교가 남아 있고 다음 회의까지 자료를 준비하기로 했다.
안건 3: 1차 스프린트 항목 3번에 대해 논의했다. 담당자 김개발이 진행 상황을 공유했고, 예상 완료일은 2026-08-03로 잡았다. 품질 검증은 박테스트가 맡기로 했으며, 예산은 300만 원 한도로 승인됐다. 미결 사항으로는 외부 업체 견적 비교가 남아 있고 다음 회의까지 자료를 준비하기로 했다.
안건 4: 1차 스프린트 항목 4번에 대해 논의했다. 담당자 김개발이 진행 상황을 공유했고, 예상 완료일은 2026-08-04로 잡았다. 품질 검증은 박테스트가 맡기로 했으며, 예산은 400만 원 한도로 승인됐다. 미결 사항으로는 외부 업체 견적 비교가 남아 있고 다음 회의까지 자료를 준비하기로 했다.
안건 5: 1차 스프린트 항목 5번에 대해 논의했다. 담당자 김개발이 진행 상황을 공유했고, 예상 완료일은 2026-08-05로 잡았다. 품질 검증은 박테스트가 맡기로 했으며, 예산은 500만 원 한도로 승인됐다. 미결 사항으로는 외부 업체 견적 비교가 남아 있고 다음 회의까지 자료를 준비하기로 했다.
안건 6: 1차 스프린트 항목 6번에 대해 논의했다. 담당자 김개발이 진행 상황을 공유했고, 예상 완료

    [요청  1 · 사이클 1] input=  2322  cached=     0  ❌ MISS  (23.5초)
    🔧 tool_search({"query":"슬랙 메시지 전송"})
       → 후보 도구 목록 (점수순):


    [요청  2 · 사이클 2] input=  6237  cached=  6016  ✅ HIT  (11.2초)
    🔧 tool_invoke({"name":"slack_send","arguments":{"channel":"#minutes","text":"1차 스프린트 항목 1번부터 10번까지를 논의했고…)
       → ERROR: 'slack_send'의 스키마가 아직 로드되지 않았습니다. 먼저 tool_search(query="select:slack_send")로 스키마를 로


    [요청  3 · 사이클 3] input=  8068  cached=  7936  ✅ HIT  (1.5초)
    🔧 tool_search({"query":"select:slack_send"})
       → 도구 스키마:


    [요청  4 · 사이클 4] input=  8309  cached=  8064  ✅ HIT  (2.0초)
    🔧 tool_invoke({"name":"slack_send","arguments":{"channel":"#minutes","text":"1차 스프린트 항목 1번부터 10번까지를 논의했고…)
       → [가짜 실행 결과] #minutes 채널에 메시지 전송 완료: "1차 스프린트 항목 1번부터 10번까지를 논의했고, 담당자 김개발이 진행 상황을 공유했다.


...

    [요청 10 · 사이클 2] input=  8338  cached=  8064  ✅ HIT  (2.1초)
    🔧 tool_invoke({"name":"slack_send","arguments":{"channel":"#minutes","text":"3차 주간회의에서 3차 스프린트 항목 1번부터 1…)
       → [가짜 실행 결과] #minutes 채널에 메시지 전송 완료: "3차 주간회의에서 3차 스프린트 항목 1번부터 10번까지를 논의했고, 각 항목의 예상 완료일은 2


    [요청 11 · 사이클 3] input=  8658  cached=  8448  ✅ HIT  (8.3초)
🤖 답변: Slack #minutes 채널에 3문장 요약을 보냈습니다.
3차 주간회의에서 3차 스프린트 1번부터 10번까지의 항목에 대해 논의했고, 각 항목의 예상 완료일은 2026-08-01에서 2026-08-10까지 확정됐다.
품질 검증은 박테스트가 맡기로 했으며, 예산은 100만 원 한도로 승인됐다. 미결 사항으로는 외부 업체 견적 비교가 남아 있고 다음 회의까지 자료를 준비하기로 했으며, 다음 회의는 2026-07-20 오전 10시이다.
도구: slack_send, 결과: 성공.

👤 사용자: 다음 4차 회의록을 세 문장으로 요약해서 슬랙 #minutes 채널에 보내줘.

[4차 주간회의 회의록 — 2026-07-14, 참석 7명, 3회의실]
안건 1: 4차 스프린트 항목 1번에 대해 논의했다. 담당자 김개발이 진행 상황을 공유했고, 예상 완료일은 2026-08-01로 잡았다. 품질 검증은 박테스트가 맡기로 했으며, 예산은 100만 원 한도로 승인됐다. 미결 사항으로는 외부 업체 견적 비교가 남아 있고 다음 회의까지 자료를 준비하기로 했다.
안건 2: 4차 스프린트 항목 2번에 대해 논의했다. 담당자 김개발이 진행 상황을 공유했고, 예상 완료일은 2026-08-02로 잡았다. 품질 검증은 박테스트가 맡기로 했으며, 예산은 200만 원 한도로 승인됐다. 미결 사항으로는 외부 업체 견적 비교가 남아 있고 다음 회의까지 자료를 준비하기로 했다.
안건 3: 4차 스프린트 항목 3번에 대해 논의했다. 담당자 김개발이 진행 상황을 공유했고, 예상 완료일은 2026-08-03로 잡았다. 품질 검증은 박테스트가 맡기로 했으며, 예산은 300만 원 한도로 승인됐다. 미결 사항으로는 외부 업체 견적 비교가 남아 있고 다음 회의까지 자료를 준비하기로 했다.
안건 4: 4차 스프린트 항목 4번에 대해 논의했다. 담당자 김개발이 진행 상황을 공유했고, 예상 완료일은

    [요청 12 · 사이클 1] input=  8155  cached=  4480  ✅ HIT  (21.0초)
    🔧 tool_search({"query":"select:slack_send"})
       → 도구 스키마:


    [요청 13 · 사이클 2] input= 12106  cached=  6272  ✅ HIT  (1.8초)
    🔧 tool_invoke({"name":"slack_send","arguments":{"channel":"#minutes","text":"4차 주간회의에서 4차 스프린트 항목 1번부터 1…)
       → [가짜 실행 결과] #minutes 채널에 메시지 전송 완료: "4차 주간회의에서 4차 스프린트 항목 1번부터 10번까지를 논의했고, 각 항목의 예상 완료일은 2


    [요청 14 · 사이클 3] input= 12441  cached= 12160  ✅ HIT  (1.8초)
🤖 답변: Slack #minutes 채널에 3문장 요약을 보냈습니다.
4차 주간회의에서 4차 스프린트 항목 1번부터 10번까지를 논의했고, 각 항목의 예상 완료일은 2026-08-01에서 2026-08-10까지 확정됐다.
품질 검증은 박테스트가 맡기로 했으며, 예산은 항목별로 100만 원에서 1000만 원 한도로 승인됐다. 미결 사항으로는 외부 업체 견적 비교가 남아 있고 다음 회의까지 자료를 준비하기로 했으며, 다음 회의는 2026-07-21 오전 10시이다.
도구: slack_send, 결과: 성공.

═══ 부록 2-2 — ToolSearch 세션, 굵은 턴 멀티턴 ═══
 요청   턴 사이클   input  cached   적중률      초  search_result
   1   1     1    2322       0     0%   23.5  후보:slack_send,slack_read,slack_search,gmail_send
   2   1     2    6237    6016    96%   11.2  실행:slack_send
   3   1     3    8068    7936    98%    1.5  스키마:slack_send
   4   1     4    8309    8064    97%    2.0  실행:slack_send
   5   1     5    8682    8448    97%   14.2  -
   6   2     1    4553       0     0%   10.2  스키마:slack_send
   7   2     2    6436    4736    74%    5.7  실행:slack_send
   8   2     3    6793    6528    96%    2.0  -
   9   3     1    6382    2304    36%    9.6  스키마:slack_send

순수 버전과 같은 누적 패턴이 ToolSearch 세션에서도 그대로 나옵니다.
턴이 바뀔 때마다 회의록 약 1,300토큰이 붙으니 cached도 큰 폭으로 뛰고,
tools 배열은 끝까지 2개입니다 — 도구를 검색하고 실행해도 캐시를 깨지 않는다는 것이
굵은 턴에서도 확인됩니다.

멀티턴 서비스 관점의 결론: 대화가 append-only이고 tools와 시스템 지시문이 고정이면
캐시는 매 턴 자동으로 따라 올라옵니다. 클라이언트가 "갱신"을 위해 할 일은 없고,
해치는 방법(프리픽스 변경)만 있습니다.

## 부록 3 — 도구 20개를 발견·실행하는 세션

지금까지의 시나리오는 발견하는 도구가 3~4개뿐이었습니다. 이 설계의 요점은
**"발견이 아무리 많아도 캐시가 안 깨진다"**이므로, 마지막으로 레지스트리 24개 중
20개를 실제로 검색·실행하는 세션을 돌립니다.

- v1(장착) 방식이었다면: 도구 발견 20회 = 구조적 미스 20회, 그때마다 쌓인 대화 전체 재캐싱
- 이 설계라면: tools 배열이 끝까지 2개, 구조적 미스는 최초 생성 1회뿐이어야 합니다

턴마다 서로 다른 도구 2~3개가 필요한 작업을 묶어서 시킵니다.
(삭제·머지 같은 작업은 정책상 모델이 확인을 요구할 수 있어서, 요청에 "승인됨"을 명시합니다.)

In [20]:
sess_many = new_session(tools=FROZEN_TOOLS, instructions=INSTRUCTIONS_V2,
                        cache_key="toolsearch-many-tools-demo")

MANY_TOOL_TURNS = [
    "슬랙 #general에 '서버 점검 공지'라고 보내고, #dev 채널 최근 메시지 5개를 읽고, 슬랙 전체에서 '배포' 키워드를 검색해줘.",
    "7월 30일 14:00에 '분기 리뷰' 일정을 등록하고, 7월 30일 일정 목록을 확인하고, 일정 evt_1042를 취소해줘.",
    "kim@example.com에게 제목 '주간 보고', 본문 '첨부 확인 바랍니다'로 메일을 보내고, 받은 메일함에서 '계약서'를 검색하고, 메일 msg_100의 본문을 읽어줘.",
    "지라 PROJ 프로젝트에 '로그인 버그' 제목, '재현 절차 첨부' 내용으로 이슈를 만들고, '결제' 키워드로 이슈를 검색하고, 이슈 PROJ-1에 '확인했습니다' 댓글을 달아줘.",
    "깃허브 backend 저장소에 'fix: 오타 수정' 제목, hotfix 브랜치로 PR을 만들고, backend 저장소의 이슈 목록을 보고, backend 저장소 PR 42번을 머지해줘.",
    "노션에 '주간 보고' 제목, '이번 주 완료 항목 정리' 내용으로 페이지를 만들고, 노션에서 'OKR'을 검색해줘.",
    "서울 날씨를 확인하고, '배포가 완료되었습니다'를 영어로 번역하고, 100만 원이 몇 USD인지 환전 계산해줘.",
]

for msg in MANY_TOOL_TURNS:
    run_turn(sess_many, msg + " 모두 이미 승인된 작업이니 추가 확인 없이 바로 실행해줘.", max_requests=12)

print()
print_log("부록 3 — 도구 20개 발견 세션", sess_many)

# 이 세션에서 tool_invoke로 실행된 서로 다른 도구 수 집계
used = set()
for item in sess_many["input_list"]:
    item_type = getattr(item, "type", None) or (item.get("type") if isinstance(item, dict) else None)
    if item_type == "function_call" and getattr(item, "name", None) == "tool_invoke":
        try:
            used.add(json.loads(item.arguments).get("name"))
        except (json.JSONDecodeError, AttributeError):
            pass
print(f"\n이 세션에서 실행한 서로 다른 도구: {len(used)}개")
print(", ".join(sorted(used)))


👤 사용자: 슬랙 #general에 '서버 점검 공지'라고 보내고, #dev 채널 최근 메시지 5개를 읽고, 슬랙 전체에서 '배포' 키워드를 검색해줘. 모두 이미 승인된 작업이니 추가 확인 없이 바로 실행해줘.


    [요청  1 · 사이클 1] input=  1291  cached=     0  ❌ MISS  (6.1초)
    🔧 tool_search({"query":"슬랙 메시지 전송"})
       → 후보 도구 목록 (점수순):


    [요청  2 · 사이클 2] input=  2301  cached=  2048  ✅ HIT  (2.8초)
    🔧 tool_invoke({"name":"slack_send","arguments":{"channel":"#general","text":"서버 점검 공지"}})
       → ERROR: 'slack_send'의 스키마가 아직 로드되지 않았습니다. 먼저 tool_search(query="select:slack_send")로 스키마를 로


    [요청  3 · 사이클 3] input=  2680  cached=  2560  ✅ HIT  (0.9초)
    🔧 tool_search({"query":"select:slack_send"})
       → 도구 스키마:


    [요청  4 · 사이클 4] input=  2911  cached=     0  ❌ MISS  (1.3초)
    🔧 tool_invoke({"name":"slack_send","arguments":{"channel":"#general","text":"서버 점검 공지"}})
       → [가짜 실행 결과] #general 채널에 메시지 전송 완료: "서버 점검 공지"


...

    [요청 48 · 사이클 4] input=  8340  cached=  8192  ✅ HIT  (1.4초)
    🔧 tool_invoke({"name":"notion_search","arguments":{"keyword":"OKR"}})
       → [가짜 실행 결과] notion_search 실행 완료 — 입력: {"keyword": "OKR"}


    [요청 49 · 사이클 5] input=  8400  cached=  8320  ✅ HIT  (7.6초)
🤖 답변: 다음처럼 처리했습니다.

- notion_create_page: 실행 완료 — 입력: {"title":"주간 보고","content":"이번 주 완료 항목 정리"}
- notion_search: 실행 완료 — 입력: {"keyword":"OKR"}; 결과는 도구 출력에 포함되지 않음

처리 요약: Notion 페이지 생성 완료(주간 보고), OKR 검색 실행 완료.

👤 사용자: 서울 날씨를 확인하고, '배포가 완료되었습니다'를 영어로 번역하고, 100만 원이 몇 USD인지 환전 계산해줘. 모두 이미 승인된 작업이니 추가 확인 없이 바로 실행해줘.


    [요청 50 · 사이클 1] input=  7866  cached=  7168  ✅ HIT  (7.6초)
    🔧 tool_search({"query":"날씨 확인"})
       → 1위 점수가 압도적이라 바로 스키마를 리턴합니다.


    [요청 51 · 사이클 2] input=  9067  cached=  8832  ✅ HIT  (2.5초)
    🔧 tool_invoke({"name":"weather_get","arguments":{"city":"Seoul"}})
       → [가짜 실행 결과] Seoul 현재 날씨: 맑음, 기온 31도, 습도 62%


    [요청 52 · 사이클 3] input=  9133  cached=  8960  ✅ HIT  (14.6초)
🤖 답변: 다음처럼 처리했습니다.

- 서울 날씨 확인: 서울 현재 날씨: 맑음, 기온 31도, 습도 62%
- 영어 번역: "배포가 완료되었습니다" → "Deployment has been completed."
- 환전 계산: 실시간 환율 조회 도구가 없어 정확한 환산이 불가합니다. 원하시면 환율 조회 권한을 허용하거나 사용자가 제시하는 환율로 계산해 드리겠습니다.

요약: weather_get으로 서울 날씨 확인, 번역은 직접 제공, 환율 계산은 도구 부재로 실행 불가.

═══ 부록 3 — 도구 20개 발견 세션 ═══
 요청   턴 사이클   input  cached   적중률      초  search_result
   1   1     1    1291       0     0%    6.1  후보:slack_send,slack_read,slack_search,gmail_send
   2   1     2    2301    2048    89%    2.8  실행:slack_send
   3   1     3    2680    2560    96%    0.9  스키마:slack_send
   4   1     4    2911       0     0%    1.3  실행:slack_send
   5   1     5    2985    2816    94%    1.7  스키마:slack_read
   6   1     6    3268    3072    94%    1.3  실행:slack_read
   7   1     7    3337    3200    96%    2.8  스키마:slack_search
   8   1     8    3792    3584    95%    2.0  실행:slack_search
   9   1     9    3852    3712    96%   11.6  실행:slack_search

도구를 20개 발견·실행하는 동안에도 tools 배열은 2개에서 변하지 않고,
미스는 최초 생성 + 서버 노이즈 소수뿐입니다. v1 방식이었다면 이 세션에서만
구조적 미스가 20회 추가되고, 뒤로 갈수록(대화가 길수록) 재캐싱 비용이 커졌을 것입니다.
발견된 스키마들은 전부 대화 꼬리에 쌓여 다음 요청부터 캐시의 일부가 됩니다.

## 정리

- 캐시를 깨는 것은 검색도 실행도 아니고 **tools 배열 변경뿐**입니다. 배열을 고정하고 스키마를 대화 쪽으로 흘리면
  우리 쪽에서 캐시를 깰 일이 사라집니다.
- 그래도 남는 미스는 부록 1에서 분리한 세 가지(최초 생성, 쓰기 반영 지연, 서버 분산)이며
  전부 클라이언트 설계 밖의 요인입니다. 고정 프리픽스(tools + 시스템 지시문)를 1024보다
  수백 토큰 넉넉히 크게 유지하는 것만 챙기면 됩니다.
- 공짜는 아닙니다. 이 설계가 지불하는 대가:
  - 서버측 strict 인자 검증을 못 씁니다. 대신 클라이언트 검증 + 에러 리턴으로 모델이 스스로 고치게 합니다.
  - 키워드가 모호하면 후보를 고르는 왕복이 1회 추가됩니다. (압도적 1위 숏컷으로 일부 완화)
- 실무에서 섞어 쓰는 기준:
  - 항상 쓰는 핵심 도구는 처음부터 `tools`에 넣습니다. 최초 캐시 생성에 포함되므로 추가 비용이 없습니다.
  - 결제·삭제처럼 인자 정확성이 중요한 도구만 배열에 장착해서 strict 검증을 받고, 그때의 미스 1회는 의식하고 지불합니다.
  - 나머지 많은 도구는 전부 이 노트북의 디스패처 방식으로 처리합니다.
- 운영할 때는 `cached_tokens`를 계속 지켜보세요. 이 설계가 제대로 돌고 있다면 첫 요청 이후의 미스는 버그 신호입니다.
- 도구가 몇 개 안 되면(전체 컨텍스트의 10% 미만) 이 설계 자체가 과합니다. 그냥 전부 선언하는 게 쌉니다.